# **Demo: Chain of Thought (CoT) Prompting with Groq**

### **Steps to Follow:**
- **Step 1:** Set Up Groq Authentication and the API Client
- **Step 2:** Define a Function to Get Completion
- **Step 3:** Evaluate Standard Direct Prompting (Baseline Without CoT)
- **Step 4:** Implement Zero-Shot Chain-of-Thought ("Let's think step by step")
- **Step 5:** Implement Few-Shot Chain-of-Thought for Complex Engineering Logic

### **Step 1: Set Up Groq Authentication and the API Client**
- Create a key in the [Groq Console](https://console.groq.com/keys).
- Set `GROQ_API_KEY` in your environment or add it to Google Colab Secrets and grant notebook access. If no key is available there, enter it at the hidden prompt.
- Installs the OpenAI Python SDK and points it at Groq's API. No Hugging Face account, OpenAI key, or GPU is required.

In [8]:
%pip install -q openai

import os
from getpass import getpass

from openai import OpenAI

groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        try:
            groq_api_key = userdata.get("GROQ_API_KEY")
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            pass

if not groq_api_key:
    groq_api_key = getpass("Groq API key: ").strip()
if not groq_api_key:
    raise ValueError("A Groq API key is required.")

client = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1",
)
print("Groq client ready.")

Note: you may need to restart the kernel to use updated packages.
Groq client ready.


### **Step 2: Define a Function to Get Completion**
- Checks Groq's model list and prefers `openai/gpt-oss-20b`, then `openai/gpt-oss-120b`, then a supported Llama chat model if listed.
- To use a different model, set `GROQ_MODEL` to an exact chat model ID from Groq's list before running the helper cell. `gpt-4o-mini` is not served by Groq.
- Lets each example set its own output limit through `max_tokens` and uses low reasoning effort for the GPT-OSS models.
- Sets `temperature=0` to reduce variability and returns the model's reply as text.

In [9]:
available_models = {model.id for model in client.models.list().data}
preferred_models = (
    "openai/gpt-oss-20b",
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "llama-3.1-8b-instant",
)
groq_model = os.environ.get("GROQ_MODEL", "").strip()
if not groq_model:
    groq_model = next((name for name in preferred_models if name in available_models), "")
if groq_model not in available_models:
    listed_models = ", ".join(sorted(available_models)) or "none"
    raise ValueError(
        f"No usable Groq model selected ({groq_model or 'no preferred model listed'}). "
        f"Set GROQ_MODEL to a chat model from this list: {listed_models}"
    )

def get_completion(prompt, max_tokens=250, model=None):
    selected_model = groq_model if model is None else model
    options = {"reasoning_effort": "low"} if selected_model in ("openai/gpt-oss-20b", "openai/gpt-oss-120b") else {}
    response = client.chat.completions.create(
        model=selected_model,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=max_tokens,
        temperature=0,
        **options,
    )
    return (response.choices[0].message.content or "").strip()

print(f"Completion helper ready. Using Groq model: {groq_model}")

Completion helper ready. Using Groq model: openai/gpt-oss-20b


### **Step 3: Evaluate Standard Direct Prompting (Baseline Without CoT)**
- Direct prompting asks the model for an immediate final answer without reasoning steps.
- In complex engineering capacity and latency calculations, small-to-medium models often fail or hallucinate when forced to output a final number directly.

In [10]:
# Baseline task: Server capacity planning
direct_prompt = """
A microservice cluster processes 1,800 requests per minute.
Each server instance can reliably handle 15 requests per second.
We need to maintain a 20% capacity headroom for traffic spikes.
How many server instances must be provisioned?

Answer directly with just the final number.
"""

print("--- Baseline: Direct Prompting (No CoT) ---")
print(get_completion(direct_prompt, max_tokens=200))

--- Baseline: Direct Prompting (No CoT) ---
3


### **Step 4: Implement Zero-Shot Chain-of-Thought ("Let's think step by step")**
- Appending the reasoning trigger **"Think step-by-step before providing the final answer"** instructs the model to generate intermediate calculations and state transformations.
- This substantially increases mathematical and logical accuracy without requiring handcrafted examples.

In [11]:
# Zero-Shot CoT with the step-by-step reasoning directive
zero_shot_cot_prompt = """
A microservice cluster processes 1,800 requests per minute.
Each server instance can reliably handle 15 requests per second.
We need to maintain a 20% capacity headroom for traffic spikes.
How many server instances must be provisioned?

Let's think step by step. Show your calculations clearly, then state the final count of instances.
"""

print("--- Zero-Shot Chain of Thought ---")
print(get_completion(zero_shot_cot_prompt, max_tokens=1000))

--- Zero-Shot Chain of Thought ---
**Step‑by‑step calculation**

1. **Convert the traffic to a per‑second rate**  
   \[
   1{,}800 \text{ requests/min} \;\div\; 60 \text{ s/min} = 30 \text{ requests/second}
   \]

2. **Add 20 % capacity headroom**  
   \[
   30 \text{ req/s} \times 1.20 = 36 \text{ req/s}
   \]

3. **Determine how many instances are needed**  
   Each instance can handle 15 req/s, so:
   \[
   \frac{36 \text{ req/s}}{15 \text{ req/s per instance}} = 2.4 \text{ instances}
   \]

4. **Round up to the next whole instance**  
   We cannot provision a fraction of a server, so we round up:
   \[
   \lceil 2.4 \rceil = 3 \text{ instances}
   \]

---

**Answer:** **3 server instances** must be provisioned to maintain a 20 % capacity headroom for traffic spikes.


### **Step 5: Implement Few-Shot Chain-of-Thought for Complex Engineering Logic**
- **Few-Shot CoT** provides solved reasoning traces showing the model exactly *how* to break down algorithmic runtime complexity and memory analysis.
- It establishes a strict `Reasoning Trace:` and `Conclusion:` schema for consistent automated parsing.

In [13]:
# Few-Shot CoT prompt showing step-by-step architectural evaluation
few_shot_cot_prompt = """
Analyze the time and space complexity of the given algorithm.

Example 1:
Algorithm:
def find_duplicates(arr):
    seen = set()
    duplicates = []
    for x in arr:
        if x in seen:
            duplicates.append(x)
        seen.add(x)
    return duplicates

Reasoning Trace:
1. We iterate over `arr` of size N once, performing O(1) hash set lookups and insertions per element.
2. Total iteration time is O(N).
3. The `seen` set stores at most N elements, and `duplicates` stores at most N elements.
4. Total extra memory allocated scales linearly with input size.
Conclusion: Time Complexity: O(N) | Space Complexity: O(N)

---
Current Task:
Algorithm:
def matrix_search(matrix, target):
    # matrix is N x N
    for row in matrix:
        for val in row:
            if val == target:
                return True
    return False

Reasoning Trace:
"""

print("--- Few-Shot Chain of Thought (Complexity Analysis) ---")
print(get_completion(few_shot_cot_prompt, max_tokens=500))

--- Few-Shot Chain of Thought (Complexity Analysis) ---
**Reasoning Trace**

1. **Input size**  
   - The matrix is an \(N \times N\) grid, so it contains \(N^2\) elements.  
   - Let \(M = N^2\) denote the total number of elements.

2. **Outer loop**  
   - The algorithm iterates over each row of the matrix.  
   - There are \(N\) rows, so the outer loop runs \(N\) times.

3. **Inner loop**  
   - For each row, the algorithm iterates over every value in that row.  
   - Each row has \(N\) elements, so the inner loop runs \(N\) times per row.

4. **Total iterations**  
   - The nested loops together visit every element exactly once:  
     \(N \times N = N^2 = M\) iterations.

5. **Work per iteration**  
   - Inside the inner loop we perform a single equality comparison (`val == target`) and, if true, a return.  
   - These operations are \(O(1)\) each.

6. **Worst‑case time**  
   - If the target is not present (or is at the very last position), the algorithm will examine all \(M\) el